# Qwen3.5-0.8B — нормализатор диктовки (Speech Clip)

Fine-tune **Qwen3.5-0.8B** на `train.jsonl` / `eval.jsonl` (только текст, без картинок).

**Colab:** Runtime → GPU (T4) → Run all. После ячейки Installation: **Runtime → Restart session**, затем Run all снова.

**Перед запуском:** загрузите `train.jsonl` и `eval.jsonl` из `dictation-normalizer/data/`.

**После обучения:** скачайте `qwen35_08b_norm_merged.zip` → на Mac: `./dictation-normalizer/scripts/integrate_model.sh`

> **Про архитектуру:** `unsloth/Qwen3.5-0.8B` — unified VLM (в чекпоинте есть vision-энкодер), но мы обучаем **только текстовый путь** — картинки не подаём. Для диктовки это нормально; vision-веса просто не используются.
>
> После интеграции обновите промпт в `normalizer.rs` под ChatML Qwen (не Gemma).


### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "transformers>=5.2.0" unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install "transformers>=5.2.0"
!pip install --no-deps trl==0.22.2

### Load Qwen3.5-0.8B

Unified VLM-чекпоинт, но fine-tune **только на тексте** (без изображений).
Unsloth рекомендует **bf16 LoRA**, не 4bit QLoRA для Qwen3.5.

In [ ]:
from unsloth import FastModel
import torch

max_seq_length = 2048  # 4096 если есть A100 и длинные диктовки

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # Unsloth: QLoRA 4bit для Qwen3.5 не рекомендуется
    load_in_8bit = False,
    load_in_16bit = True,   # bf16 LoRA ~3 GB VRAM на 0.8B
    full_finetuning = False,
    # token = "hf_...",  # если модель gated
)

### LoRA

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep

Формат ChatML (Qwen):

```
<|im_start|>system
...
<|im_start|>user
...
<|im_start|>assistant
...
```

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3",
)

In [ ]:
# Upload train.jsonl and eval.jsonl into Colab first (left file panel).
from datasets import load_dataset

dataset      = load_dataset("json", data_files="train.jsonl", split="train")
eval_dataset = load_dataset("json", data_files="eval.jsonl",  split="train")
print(dataset)
print(dataset[0])

In [ ]:
SYSTEM_PROMPT = (
    "Ты нормализуешь русскую голосовую диктовку разработчика. Английские технические "
    "термины, имена файлов, бренды и аббревиатуры, записанные русскими буквами по звучанию, "
    "замени на правильное написание (коммит -> commit, карго томл -> @Cargo.toml, зум -> Zoom). "
    "Имена файлов пиши через собачку @. Русские слова не переводи и не меняй. "
    "Меняй только термины, всё остальное оставляй как есть."
)

def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": example["in"]},
            {"role": "assistant", "content": example["out"]},
        ]
    }

dataset      = dataset.map(convert_to_chatml)
eval_dataset = eval_dataset.map(convert_to_chatml)

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
            enable_thinking = False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset[min(100, len(dataset)-1)]

<a name="Train"></a>
### Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.0f}s training")
print(f"Peak reserved memory = {used_memory} GB (LoRA +{used_memory_for_lora} GB)")

<a name="Inference"></a>
### Inference

Для нормализатора: **temperature=0**, без thinking.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "Проверь, отработал лискрипт, который называется cleanupdatabase.py"},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens = 256,
    temperature = 0.0,
    do_sample = False,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

### Eval quick check (exact-match на eval.jsonl)

In [ ]:
import json

def normalize(text):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    out = model.generate(
        **tokenizer(prompt, return_tensors="pt").to("cuda"),
        max_new_tokens=256,
        temperature=0.0,
        do_sample=False,
    )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # ответ после последнего assistant
    if "assistant" in full:
        full = full.split("assistant")[-1]
    return full.strip()

rows = [json.loads(l) for l in open("eval.jsonl", encoding="utf-8") if l.strip()]
ok = 0
for i, row in enumerate(rows[:30], 1):
    pred = normalize(row["in"])
    match = pred.strip() == row["out"].strip()
    ok += int(match)
    if not match:
        print(f"[{i}] IN : {row['in'][:80]}")
        print(f"     WANT: {row['out'][:80]}")
        print(f"     GOT : {pred[:80]}\n")
print(f"exact-match (first 30): {ok}/30")

<a name="Save"></a>
### Save merged model for Mac

In [ ]:
model.save_pretrained_merged(
    "qwen35_08b_norm_merged",
    tokenizer,
    save_method = "merged_16bit",
)

import shutil
shutil.make_archive("qwen35_08b_norm_merged", "zip", "qwen35_08b_norm_merged")
print("Done -> download qwen35_08b_norm_merged.zip")

### Optional: GGUF (если integrate_model.sh не подходит)

```python
model.save_pretrained_gguf(
    "qwen35_08b_norm_gguf",
    tokenizer,
    quantization_method = "Q4_K_M",
)
```

Документация: [Unsloth Qwen3.5 fine-tune](https://unsloth.ai/docs/models/qwen3.5/fine-tune)
